# AnyTime Universal Benchmark

Run early-exit benchmark for any AnyTime model. Switch by changing **one import line** below.

Captures **4 dimensions** for every run: quality, memory, latency, energy.

Per run dir contains:
- `hw_results.json` — latency + memory + energy (no quality calc)
- `quality_results.json` — accuracy / F1 / mAP / ROUGE (no HW measure)

Outputs:
- `logs/benchmark/{model}/...` — per-run JSONs
- `results/{model}/` — `latency.csv`, `energy.csv`, `quality.csv`, `hardware.csv`

## 1. Setup

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path('.').resolve()
sys.path.insert(0, str(REPO_ROOT))
print('repo root:', REPO_ROOT)

In [ ]:
# HF login (only needed if pulling from a private repo)
from shared import load_env
load_env()

try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN', ''))
except Exception:
    pass

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF: logged in')
else:
    print('HF: no token, public repos only')

## 2. Choose model

**Change ONE import line below** to swap which model gets benchmarked.

In [ ]:
# === SWAP THIS LINE TO PICK MODEL ===
from benchmark_config import bert as cfg
# from benchmark_config import llama  as cfg
# from benchmark_config import vision as cfg
# from benchmark_config import yolo   as cfg

print('model :', cfg.NAME)
print('family:', cfg.MODEL_FAMILY)
print('out   :', cfg.OUT_DIR)

## 3. Run full sweep

Iterates every sweep point. Each run does **2 passes**:
1. `profile_hw()` — pure HW + latency, no quality calc
2. `evaluate_quality()` — pure quality, no HW measure

Two JSONs per run: `hw_results.json` + `quality_results.json`.

In [ ]:
cfg.run_all()
# cfg.run_all(skip_quality=True)   # HW only
# cfg.run_all(skip_hw=True)        # quality only

## 4. Export CSVs (latency / energy / quality / hardware)

Auto-merges `hw_results.json` + `quality_results.json` per run dir, writes 4 CSVs.

In [ ]:
from shared import write_benchmark_csvs, average_across_tasks
from shared.averager import write_averaged_json

out_dir = Path(cfg.OUT_DIR)
csv_root = REPO_ROOT / 'results' / cfg.NAME

for task_dir in out_dir.iterdir():
    if not task_dir.is_dir():
        continue
    files = {sub.name: sub for sub in task_dir.iterdir()
             if sub.is_dir() and ((sub / 'hw_results.json').exists()
                                  or (sub / 'benchmark_results.json').exists())}
    if not files:
        continue
    baseline = next((k for k in sorted(files)
                     if 'entropy_0.0' in k or 'baseline' in k), None)
    write_benchmark_csvs(
        results_files=files,
        out_dir=csv_root / task_dir.name,
        baseline_key=baseline,
        method_order=sorted(files.keys()),
    )

# averaged across tasks
per_task = {td.name: td for td in out_dir.iterdir() if td.is_dir()}
if per_task:
    averaged = average_across_tasks(per_task, run_pattern='*')
    write_averaged_json(averaged, csv_root / 'averaged.json')

print('CSVs written under:', csv_root)

## 5. Quick view

In [ ]:
import pandas as pd
for csv_name in ['latency.csv', 'energy.csv', 'quality.csv', 'hardware.csv']:
    for task_csv_dir in csv_root.iterdir():
        if not task_csv_dir.is_dir():
            continue
        f = task_csv_dir / csv_name
        if f.exists():
            print('===', task_csv_dir.name, '/', csv_name, '===')
            print(pd.read_csv(f).head(8))
            print()